# GoodForget-RAG Demo

A minimal notebook for running the toy retrieval comparison.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from goodforget_rag.eval import build_query_result_row, results_dataframe, summarize_results
from goodforget_rag.retrieval import goodforget_retrieve, positive_only_retrieve, vanilla_retrieve
from goodforget_rag.vectorizer import TfidfEncoder, document_to_text

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

def load_jsonl(path):
    with path.open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

documents = load_jsonl(ROOT / "experiments" / "toy_corpus.jsonl")
queries = load_jsonl(ROOT / "experiments" / "toy_queries.jsonl")

training_texts = [document_to_text(doc) for doc in documents]
for query in queries:
    training_texts.append(query["query"])
    training_texts.append(query["positive_intent"])
    training_texts.extend(query["forget_set"])

encoder = TfidfEncoder().fit(training_texts)

In [ ]:
rows = []
for query in queries:
    rows.append(build_query_result_row("Vanilla RAG", query, vanilla_retrieve(encoder, documents, query["query"], top_k=2)))
    rows.append(build_query_result_row("Positive-only RAG", query, positive_only_retrieve(encoder, documents, query["positive_intent"], top_k=2)))
    rows.append(build_query_result_row("GoodForget-RAG", query, goodforget_retrieve(
        encoder,
        documents,
        query["query"],
        query["positive_intent"],
        query["forget_set"],
        alpha=0.4,
        beta=0.8,
        gamma=1.2,
        top_k=2,
    )))

results = results_dataframe(rows)
summary = summarize_results(rows)
summary

In [ ]:
example = results[results["query_id"] == "q_password_reset"]
example[["method", "retrieved_doc_ids", "leaked_doc_ids", "utility_recall"]]